# QNN vs Kernel Pivot Scoreboard

This notebook is meant for team decision-making.

If your team has already invested heavily in QNNs, the right move is not panic-pivoting. The right move is to compare three defensible options on the same proxy task:

1. A structured QCNN-style QNN.
2. A data re-uploading QNN.
3. A quantum-kernel `QSVC` fallback.

The goal is to decide what you should present as the main method and what should become supporting evidence.

## 1. Imports

Everything is kept in one notebook so the results are directly comparable.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import zz_feature_map
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms.optimizers import COBYLA, SPSA
from qiskit_machine_learning.algorithms import QSVC
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.neural_networks import SamplerQNN

SEED = 8398
np.random.seed(SEED)

## 2. Helpers

This includes the shared industrial proxy dataset plus builders for the QCNN-style and data re-uploading circuits.

In [ ]:
def generate_microdefect_latents(n_samples=80, seed=SEED + 100):
    """Synthetic proxy for calibrated industrial texture features.

    Think of each row as four compact features produced by an inspection front-end:
    two local texture phase measurements and two orientation/defect-coupling scores.
    The label depends on coupled factors, not a single obvious threshold.
    """
    rng = np.random.default_rng(seed)
    X = rng.uniform(0.0, 2 * np.pi, size=(n_samples, 4))
    score = np.sin(X[:, 0]) * np.cos(X[:, 1]) + np.sin(X[:, 2] + X[:, 3])
    y = (score > np.median(score)).astype(int)
    return X, y, score


def latents_to_microtexture_images(X, size=8, seed=SEED):
    """Render latent texture factors as small grayscale inspection patches."""
    rng = np.random.default_rng(seed)
    grid = np.linspace(0, 2 * np.pi, size)
    rr, cc = np.meshgrid(grid, grid, indexing="ij")
    images = []

    for x in X:
        img = (
            0.50
            + 0.16 * np.sin(rr + x[0])
            + 0.13 * np.cos(cc + x[1])
            + 0.10 * np.sin(rr + cc + x[2])
            + 0.08 * np.cos(rr - cc + x[3])
        )
        img += rng.normal(0.0, 0.025, size=(size, size))
        images.append(np.clip(img, 0.0, 1.0))

    return np.array(images)


def show_microtextures(images, labels, scores=None, n_show=12, cols=6):
    order = np.arange(min(n_show, len(images)))
    rows = int(np.ceil(len(order) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.7))
    axes = np.ravel(axes)

    for ax, idx in zip(axes, order):
        ax.imshow(images[idx], cmap="gray_r", vmin=0, vmax=1)
        label = "DEFECT" if labels[idx] else "clean"
        suffix = "" if scores is None else f"\nscore={scores[idx]:.2f}"
        ax.set_title(f"#{idx} {label}{suffix}", fontsize=8, color="crimson" if labels[idx] else "black")
        ax.set_xticks([])
        ax.set_yticks([])

    for ax in axes[len(order):]:
        ax.axis("off")

    fig.tight_layout()
    plt.show()


def evaluate_predictions(name, y_true, y_pred, *, show_confusion=True):
    print(f"\n{name}")
    print("balanced accuracy:", balanced_accuracy_score(y_true, y_pred))
    print("defect F1 score  :", f1_score(y_true, y_pred, zero_division=0))
    print(classification_report(y_true, y_pred, target_names=["clean", "defect"], zero_division=0))

    if show_confusion:
        ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=["clean", "defect"])
        plt.title(name)
        plt.show()


def evaluate_classifier(name, model, X_eval, y_eval, *, show_confusion=True):
    y_pred = model.predict(X_eval)
    evaluate_predictions(name, y_eval, y_pred, show_confusion=show_confusion)
    return y_pred


def balanced_subset(X_data, y_data, n_per_class=18, seed=SEED):
    rng = np.random.default_rng(seed)
    chosen = []
    for cls in np.unique(y_data):
        cls_idx = np.flatnonzero(y_data == cls)
        take = min(n_per_class, len(cls_idx))
        chosen.extend(rng.choice(cls_idx, size=take, replace=False).tolist())
    rng.shuffle(chosen)
    return X_data[chosen], y_data[chosen]


def readout_q0(measured_integer):
    """Use qubit 0 as the binary classifier readout."""
    return measured_integer & 1


def parity_readout(measured_integer):
    """Map a measured computational-basis state to class 0 or 1 by parity."""
    return bin(measured_integer).count("1") % 2



def build_qcnn_style_circuit(n_qubits=4):
    """A compact QCNN-style circuit for four texture features.

    This is not a full image CNN. It is a small structured QNN inspired by QCNN
    motifs: local convolutional pair interactions, pooling-like information
    concentration, then a final interaction between pooled channels.
    """
    if n_qubits != 4:
        raise ValueError("This compact demo is written for exactly four qubits.")

    x = ParameterVector("x", n_qubits)
    theta = ParameterVector("theta", 10)
    qc = QuantumCircuit(n_qubits)

    for q in range(n_qubits):
        qc.ry(x[q], q)

    # Shared local convolution on pairs (0, 1) and (2, 3).
    qc.cx(0, 1)
    qc.ry(theta[0], 1)
    qc.rz(theta[1], 0)
    qc.cx(1, 0)
    qc.ry(theta[2], 0)

    qc.cx(2, 3)
    qc.ry(theta[0], 3)
    qc.rz(theta[1], 2)
    qc.cx(3, 2)
    qc.ry(theta[2], 2)

    # Pooling-like interactions: concentrate each pair into qubits 0 and 2.
    qc.cx(1, 0)
    qc.ry(theta[3], 0)
    qc.cz(1, 0)

    qc.cx(3, 2)
    qc.ry(theta[4], 2)
    qc.cz(3, 2)

    # Final convolution/readout layer between pooled channels.
    qc.cx(0, 2)
    qc.ry(theta[5], 2)
    qc.rz(theta[6], 0)
    qc.cx(2, 0)
    qc.ry(theta[7], 0)
    qc.ry(theta[8], 2)
    qc.cz(0, 2)
    qc.ry(theta[9], 0)

    return qc, list(x), list(theta)



def build_data_reuploading_circuit(n_qubits=4, layers=1):
    x = ParameterVector("x", n_qubits)
    theta = ParameterVector("theta", layers * n_qubits * 2)
    qc = QuantumCircuit(n_qubits)

    t = 0
    for layer in range(layers):
        for q in range(n_qubits):
            qc.ry(x[q], q)
            qc.rz(x[(q + 1) % n_qubits], q)

        for q in range(n_qubits):
            qc.ry(theta[t], q)
            t += 1
            qc.rz(theta[t], q)
            t += 1

        for q in range(n_qubits - 1):
            qc.cx(q, q + 1)
        qc.cx(n_qubits - 1, 0)

    return qc, list(x), list(theta)


def qnn_classifier_from_circuit(qc, input_params, weight_params, interpret, maxiter=40, seed=SEED):
    sampler_qnn = SamplerQNN(
        circuit=qc,
        sampler=StatevectorSampler(seed=seed),
        input_params=input_params,
        weight_params=weight_params,
        interpret=interpret,
        output_shape=2,
    )
    rng = np.random.default_rng(seed)
    initial_point = rng.uniform(-0.2, 0.2, size=len(weight_params))
    return NeuralNetworkClassifier(
        neural_network=sampler_qnn,
        optimizer=COBYLA(maxiter=maxiter),
        initial_point=initial_point,
    )

## 3. Generate the Shared Benchmark

This is still a synthetic proxy. Its purpose is to test method behavior before you decide what real-world story to lead with.

In [ ]:
N_SAMPLES = 80
X, y, scores = generate_microdefect_latents(n_samples=N_SAMPLES, seed=SEED + 100)
images = latents_to_microtexture_images(X, size=8, seed=SEED)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y,
)

print("feature shape:", X.shape)
print("train/test:", X_train.shape, X_test.shape)
print("class balance:", {"clean": int((y == 0).sum()), "defect": int((y == 1).sum())})
show_microtextures(images, y, scores=scores, n_show=12, cols=6)

## 4. Baseline Score Table

The scoreboard uses balanced accuracy and defect F1. In rare-defect settings, these are more meaningful than ordinary accuracy.

In [ ]:
score_rows = []


def add_score(name, y_true, y_pred, family):
    score_rows.append({
        "model": name,
        "family": family,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "defect_f1": f1_score(y_true, y_pred, zero_division=0),
    })


classical_models = {
    "Linear SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="linear", class_weight="balanced", random_state=SEED)),
    ]),
    "RBF SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="rbf", class_weight="balanced", random_state=SEED)),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=SEED,
    ),
}

for name, model in classical_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    add_score(name, y_test, pred, "classical")

score_rows

## 5. QCNN-Style QNN

This is the best QNN-centered story if it performs well: structured, low-parameter, local-to-global.

In [ ]:
X_qnn_train, y_qnn_train = balanced_subset(X_train, y_train, n_per_class=18, seed=SEED)

qcnn_circuit, qcnn_inputs, qcnn_weights = build_qcnn_style_circuit(n_qubits=4)
qcnn_classifier = qnn_classifier_from_circuit(
    qcnn_circuit,
    qcnn_inputs,
    qcnn_weights,
    interpret=readout_q0,
    maxiter=40,
    seed=SEED,
)
qcnn_classifier.fit(X_qnn_train, y_qnn_train)
qcnn_pred = qcnn_classifier.predict(X_test)
add_score("QCNN-style SamplerQNN", y_test, qcnn_pred, "QNN")
evaluate_predictions("QCNN-style SamplerQNN", y_test, qcnn_pred, show_confusion=False)

## 6. Data Re-Uploading QNN

This is the most standard QNN classifier option. It is viable, but more sensitive to readout and optimizer settings.

In [ ]:
reupload_circuit, reupload_inputs, reupload_weights = build_data_reuploading_circuit(n_qubits=4, layers=1)
reupload_classifier = qnn_classifier_from_circuit(
    reupload_circuit,
    reupload_inputs,
    reupload_weights,
    interpret=parity_readout,
    maxiter=80,
    seed=SEED,
)
reupload_classifier.fit(X_qnn_train, y_qnn_train)
reupload_pred = reupload_classifier.predict(X_test)
add_score("Data re-uploading SamplerQNN", y_test, reupload_pred, "QNN")
evaluate_predictions("Data re-uploading SamplerQNN", y_test, reupload_pred, show_confusion=False)

## 7. Quantum Kernel Fallback

This is the pivot path. If QNN training is unstable, `QSVC` gives a cleaner quantum-ML story with fewer trainable quantum parameters.

In [ ]:
feature_map = zz_feature_map(feature_dimension=4, reps=1)
quantum_kernel = FidelityQuantumKernel(feature_map=feature_map)
qsvc = QSVC(quantum_kernel=quantum_kernel, class_weight="balanced")
qsvc.fit(X_train, y_train)
qsvc_pred = qsvc.predict(X_test)
add_score("Quantum Kernel QSVC", y_test, qsvc_pred, "quantum kernel")
evaluate_predictions("Quantum Kernel QSVC", y_test, qsvc_pred, show_confusion=False)

## 8. Final Scoreboard

Use this table to decide the final presentation story.

In [ ]:
score_rows = sorted(score_rows, key=lambda row: row["balanced_accuracy"], reverse=True)

for row in score_rows:
    print(
        f"{row['model']:<32s} "
        f"family={row['family']:<14s} "
        f"balanced_acc={row['balanced_accuracy']:.3f} "
        f"defect_f1={row['defect_f1']:.3f}"
    )

best = score_rows[0]
print()
print("Current best:", best["model"], "(", best["family"], ")")

## 9. Decision Rule

Use this simple rule for the hackathon:

- If QCNN-style QNN wins or is close, lead with **structured QNN for industrial micro-defect inspection**.
- If data re-uploading wins, lead with **compact QNN for coupled texture-feature classification**.
- If `QSVC` wins or QNN results jump around across seeds, pivot to **quantum kernels** and keep QNNs as an explored path.
- If classical baselines win, present this honestly as a benchmark-and-feasibility study rather than claiming advantage.

## References and AI Tools Disclosure

This notebook was drafted with OpenAI Codex/GPT-5 assistance in the local hackathon workspace. The team should treat the code and results as AI-assisted prototypes and verify claims before presenting them.

AI-assisted notebooks in this `main_challenge` folder:

- `01_defect_classical_hardness_audit.ipynb`
- `02_quantum_kernel_teacher_defects.ipynb`
- `03_projected_quantum_kernel_features.ipynb`
- `04_qnn_vs_kernel_bakeoff.ipynb`
- `05_qcnn_industrial_microdefects.ipynb`
- `06_data_reuploading_qnn_microdefects.ipynb`
- `07_qnn_kernel_pivot_scoreboard.ipynb`
- `08_imbalanced_rare_defect_qnn.ipynb`
- `09_normal_only_anomaly_detection.ipynb`

References used for the quantum-ML direction:

- Cong, Choi, and Lukin, "Quantum convolutional neural networks," Nature Physics 15, 1273-1278 (2019): https://www.nature.com/articles/s41567-019-0648-8
- Pérez-Salinas et al., "Data re-uploading for a universal quantum classifier," Quantum 4, 226 (2020): https://doi.org/10.22331/q-2020-02-06-226 and https://arxiv.org/abs/1907.02085
- McClean et al., "Barren plateaus in quantum neural network training landscapes," Nature Communications 9, 4812 (2018): https://www.nature.com/articles/s41467-018-07090-4
- Havlíček et al., "Supervised learning with quantum-enhanced feature spaces," Nature 567, 209-212 (2019): https://www.nature.com/articles/s41586-019-0980-2
- Schölkopf et al., "Estimating the Support of a High-Dimensional Distribution," Neural Computation 13(7), 1443-1471 (2001): https://doi.org/10.1162/089976601750264965
- Ngairangbam, Spannowsky, and Takeuchi, "Anomaly detection in high-energy physics using a quantum autoencoder," Physical Review D 105, 095004 (2022): https://arxiv.org/abs/2112.04958
- Quantum Patch-Based Autoencoder for Anomaly Segmentation: https://arxiv.org/abs/2404.17613
- Quantum Machine Learning Algorithms for Anomaly Detection: a Review: https://arxiv.org/abs/2408.11047
- Qiskit Machine Learning `SamplerQNN` documentation: https://qiskit-community.github.io/qiskit-machine-learning/stubs/qiskit_machine_learning.neural_networks.SamplerQNN.html